In [ ]:
import cobra
import pandas as pd
from cobra.io import load_model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite, Gene
import numpy as np

In [ ]:
# Define nitrogen sources (replace with IDs from your model)
nitrogen_sources = ["EX_nh4_e", "EX_no3_e", "EX_no2_e"]

# Initialize a list to store results
results = []

# Iterate over nitrogen sources
for source in nitrogen_sources:
    with model:
        # Set the uptake rate for the current nitrogen source
        model.reactions.get_by_id(source).lower_bound = -10  # Allow uptake
        model.reactions.get_by_id(source).upper_bound = 0    # Prevent export

        # Set all other nitrogen sources to 0
        for other_source in nitrogen_sources:
            if other_source != source:
                model.reactions.get_by_id(other_source).lower_bound = 0
                model.reactions.get_by_id(other_source).upper_bound = 0

        # Optimize the model
        solution = model.optimize()

        # Store the results
        results.append({
            "Nitrogen Source": source,
            "Objective Value": solution.objective_value,
            "Fluxes": solution.fluxes.to_dict()  # Store all fluxes as a dictionary
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Display the DataFrame
print(results_df[["Nitrogen Source", "Objective Value"]])

# Optional: Save to a CSV for further analysis
results_df.to_csv("nitrogen_source_analysis.csv", index=False)